# LLM Alert Investigator - Data Exploration

This notebook explores the alert data and MITRE ATT&CK knowledge base.

In [ ]:
import json
import pandas as pd
from pathlib import Path

import sys
sys.path.append('..')

from src.preprocessor import AlertPreprocessor, load_alerts_from_file
from src.kb_builder import MITREKnowledgeBaseBuilder

## Load Sample Alerts

In [ ]:
attack_alerts = load_alerts_from_file('../data/sample_alerts/attack_samples.json')
benign_alerts = load_alerts_from_file('../data/sample_alerts/benign_samples.json')

print(f"Loaded {len(attack_alerts)} attack alerts")
print(f"Loaded {len(benign_alerts)} benign alerts")

## Explore Attack Alerts

In [ ]:
attack_df = pd.DataFrame(attack_alerts)
attack_df.head()

## Preprocess Alerts

In [ ]:
preprocessor = AlertPreprocessor()

normalized_attack = [preprocessor.normalize(alert) for alert in attack_alerts]
enriched_attack = [preprocessor.enrich(alert) for alert in normalized_attack]

print("Sample normalized alert:")
print(json.dumps(enriched_attack[0], indent=2))

## Build MITRE Knowledge Base

In [ ]:
kb_builder = MITREKnowledgeBaseBuilder()

kb_path = '../data/mitre_attack_kb'

if Path(kb_path).exists():
    mitre_kb = kb_builder.load(kb_path)
    print("Loaded existing MITRE KB")
else:
    mitre_kb = kb_builder.build_from_mitre()
    kb_builder.save(kb_path)
    print("Built and saved new MITRE KB")

## Test Retrieval

In [ ]:
alert_text = preprocessor.to_text(enriched_attack[0])
print("Alert text for retrieval:")
print(alert_text)
print()

relevant_techniques = mitre_kb.similarity_search(alert_text, k=3)

print("Retrieved MITRE techniques:")
for i, doc in enumerate(relevant_techniques, 1):
    print(f"\n{i}. {doc.metadata.get('technique_id')} - {doc.metadata.get('technique_name')}")
    print(f"   Tactics: {doc.metadata.get('tactics')}")
    print(f"   Content: {doc.page_content[:200]}...")

## Analyze Alert Patterns

In [ ]:
attack_indicators = []
for alert in enriched_attack:
    attack_indicators.append(alert.get('attack_indicators', {}))

indicators_df = pd.DataFrame(attack_indicators)
print("Attack indicators in sample alerts:")
indicators_df.sum().sort_values(ascending=False)

## Severity Distribution

In [ ]:
severity_counts = pd.Series([a['severity'] for a in attack_alerts]).value_counts()
severity_counts.plot(kind='bar', title='Severity Distribution')